# Deliverable 2 — Synthetic Cohort Generator
**Project:** Student Retention Early-Warning (Philippine Higher Education) · Eskwelabs Innovation Fellowship, Data Innovation Track
**Implements:** *A Two-Layer Synthetic Data Generator for Studying Student Retention Early-Warning Systems in Philippine Higher Education* (whitepaper v17; Appendix A reference instantiation)

> ⚠️ **Everything this notebook produces is synthetic.** Numbers derived from the generated data describe the behaviour of a generative model calibrated to cited *aggregate* statistics. They are **not** measurements of any real Philippine student population. The only real numbers are the inputs tagged `CALIBRATED` in §2, and they are labelled as such.

### What this notebook does
| § | Content | Whitepaper |
|---|---|---|
| 1 | Run settings (row count, seed, switches) | §4, App. B.3 |
| 2 | **Parameter registry** — every number tagged `LOCKED` / `CALIBRATED` / `ASSUMED` | §10, Table 4 |
| 3 | Generator engine: Phases 1–5a (latent state → observables → confounded assignment → hazard) | §5–9, Eq. 1–4 |
| 4 | Phase 5b calibration loop (nested bisection with common random numbers) | §4, §11, App. A.1 |
| 5 | Assemble & export the 8 tables + metadata + content hashes | §4 Table 2, App. B |
| 6 | Sanity checks (ranges, keys, correlations, leakage-relevant integrity) | §7, App. B.2 |
| 7 | Validation scorecard — every check in Table 5 | §11 |
| 8 | Deliberate edge cases, audited against the hidden latent state | §12 |
| 9 | Customisability demo (different N, no recalibration) | Project brief |
| 10 | Assumptions log, open flags, revision log | Project "definition of done" |

### Method references (where each decision is grounded)
* **Law (2015), *Simulation Modeling and Analysis*** — input-modelling, verification & validation, common random numbers, independent replications (§3–4, §7).
* **McElreath (2020) / Peters, Janzing & Schölkopf (2017)** — the latent/observed DAG, confounding, and why adjustment on assignment inputs identifies the effect (§3, §7).
* **Gelman, Hill & Vehtari (2020), *Regression and Other Stories*** — adjusted treatment-effect estimation and the ignorability assumption (§7).
* **Gelman et al. (2013), BDA3** — hierarchical structure that the two-tier region model deliberately collapses (limitation, §10).

## 1 · Run settings
These are **run settings**, not model parameters (model parameters live in §2 and carry provenance tags).

* `N_STUDENTS` — rows in `Students`; panel tables scale to roughly **N × 6** rows each (students leave over time; ≈ 5.95 terms per student at the calibrated attrition). Whitepaper default is **50,000** and the tier-level tolerance analysis in §4 of the paper was done at that size. At small N, sampling noise can exceed the ±3-pp tolerance, so §7 may flag failures that are noise, not miscalibration.
* `RECALIBRATE` — `True` re-runs the Phase 5b loop for *this* N and seed (≈ 20–30 s at N = 50,000). `False` reuses `FROZEN_CALIBRATION` (fitted at N = 50,000, seed 193) — appropriate for quick runs at other sizes.
* `WRITE_AUDIT_LATENT` — the latent state is **never exported** by default (whitepaper App. B.2). Set `True` only for a *private* audit copy.

In [ ]:
import hashlib, json, platform, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.api as sm
import scipy
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

CONFIG = {
    "N_STUDENTS": 50_000,          # <- change me: number of synthetic students
    "MASTER_SEED": 193,            # one integer master seed (App. B.3 #1)
    "RECALIBRATE": True,           # re-run Phase 5b for this N/seed
    "FROZEN_CALIBRATION": {"delta": 1.0330, "gamma0": -2.6732},   # fitted at N=50,000, seed=193
    "OUTPUT_DIR": "output",
    "EXPORT_CSV": True,
    "WRITE_AUDIT_LATENT": False,   # keep False: latent state is never exported
    "N_REPLICATES": 20,            # independent populations for Table 5 checks (>= 20 required)
    "N_BOOTSTRAP": 1000,           # student-level bootstrap draws for the naive-estimate CI
}
SPEC_VERSION, GENERATOR_VERSION = "whitepaper v17", "1.0.0"
OUT = Path(CONFIG["OUTPUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
SYNTHETIC_BANNER = ("SYNTHETIC DATA — produced by this generative model. Not real students; "
                    "not a measurement of any real population.")
def show(df, n=5):
    print("⚠", SYNTHETIC_BANNER); display(df.head(n))
print(f"N_STUDENTS={CONFIG['N_STUDENTS']:,}  seed={CONFIG['MASTER_SEED']}  recalibrate={CONFIG['RECALIBRATE']}")
if CONFIG["N_STUDENTS"] < 50_000:
    print("NOTE: N < 50,000 — Table 5 tolerances were designed at N = 50,000; small-N failures may be sampling noise.")

## 2 · Parameter registry (`LOCKED` / `CALIBRATED` / `ASSUMED`)
Whitepaper §3 *Full Traceability*: **no number enters the generator without exactly one tag.** All generator code reads constants through `P(name)`; there are no untagged literals in the generative functions. The registry is exported as `parameter_log.csv`.

* **LOCKED** — fixed by design, not open to tuning.
* **CALIBRATED** — set to reproduce a cited figure (or fitted by the Phase 5b loop to one).
* **ASSUMED** — defensible but uncited modelling choice; candidate for sensitivity analysis. *This list is the "every assumption is written down" requirement.*

Anything not given in the whitepaper itself (the non-core attribute forms, edge-case thresholds) is tagged `ASSUMED` and listed in the open-flags log in §10.

In [ ]:
TAGS = {"LOCKED", "CALIBRATED", "ASSUMED"}
PARAM = {}
def reg(name, value, tag, basis):
    assert tag in TAGS, f"{name}: invalid tag {tag}"
    assert name not in PARAM, f"{name}: duplicate"
    PARAM[name] = {"value": value, "tag": tag, "basis": basis}
def P(name): return PARAM[name]["value"]
def set_calibrated(name, value):
    assert PARAM[name]["tag"] == "CALIBRATED", f"{name} is not CALIBRATED; refusing to tune it"
    PARAM[name]["value"] = float(value)

# ---- LOCKED -------------------------------------------------------------------------------------------
reg("T_MAX", 8, "LOCKED", "1 term = 1 semester; 8 terms per 4-year cohort (WP Table 4)")
reg("intervention_effect", 0.30, "LOCKED", "True relative hazard reduction (Eq. 3); internal ground truth, NOT an empirical claim (WP §9)")
reg("tol_pp", 0.03, "LOCKED", "±3 pp Empirical-Fidelity tolerance (WP §3). Not tagged explicitly in WP -> treated as design-fixed; see open flag OF-5")
# ---- CALIBRATED (cited figures) -----------------------------------------------------------------------
reg("target_high_2022", 0.671, "CALIBRATED", "Mean of BARMM 93.4, Reg VII 60.7, Reg IX 59.5, CAR 54.9 = 67.1% (CPBRD 2025, Class-of-2022 column; EDCOM II 2025)")
reg("target_typical_2022", 0.459, "CALIBRATED", "Enrollment-weighted mean of the 13 remaining regions, Class of 2022 = 45.9% (CPBRD 2025)")
reg("national_2023_level", 0.294, "CALIBRATED", "CHED national total read as Class of 2023 (CPBRD 2025; CHED FY2026 budget presentation: 29.39%). Held-out level only")
reg("observed_ratio_2023", 1.37, "CALIBRATED", "Class-of-2023 high/typical ratio on the 3 high-tier regions with data (CAR, VII, IX): 35.7/26.0 (WP §11)")
# ---- CALIBRATED (fitted by the Phase 5b loop; initial = Appendix A pilot) ---------------------------------
reg("delta", 1.06, "CALIBRATED", "Tier shift in baseline vulnerability; fitted by Phase 5b to the high-tier target (WP App. A.1)")
reg("gamma0", -2.67, "CALIBRATED", "Hazard intercept (Eq. 3); fitted by Phase 5b to the typical-tier target (WP App. A.1)")
# ---- ASSUMED: Phase 1 --------------------------------------------------------------------------------
reg("tier_share", 0.15, "ASSUMED", "Share of students in high-attrition tier; within CPBRD-implied 14.1–18% range (WP Table 4)")
reg("work_share", 0.35, "ASSUMED", "Working-student prevalence; directionally supported only (WP Table 4)")
# ---- ASSUMED: Phase 2/3 generative forms (WP Appendix A.1) --------------------------------------------
reg("shock_prob", 0.08, "ASSUMED", "Per-term acute-shock probability (WP Table 4)")
reg("vuln_work", 0.5, "ASSUMED", "v = δ·tier + 0.5·work − 0.4·inc + 0.8·ε (WP A.1)")
reg("vuln_inc", -0.4, "ASSUMED", "Higher income proxy -> lower vulnerability (WP A.1)")
reg("vuln_eps", 0.8, "ASSUMED", "Idiosyncratic vulnerability noise (WP A.1)")
reg("grade_intercept", 72.0, "ASSUMED", "grade = 72 − 7v − 7·shock + 8ε (WP A.1)")
reg("grade_v", -7.0, "ASSUMED", "WP A.1")
reg("grade_shock", -7.0, "ASSUMED", "WP A.1")
reg("grade_noise", 8.0, "ASSUMED", "WP A.1")
reg("att_intercept", 0.88, "ASSUMED", "attendance = 0.88 − 0.06v − 0.08·shock + 0.08ε (WP A.1)")
reg("att_v", -0.06, "ASSUMED", "WP A.1")
reg("att_shock", -0.08, "ASSUMED", "WP A.1")
reg("att_noise", 0.08, "ASSUMED", "WP A.1")
reg("hold_intercept", -2.2, "ASSUMED", "hold ~ Bernoulli(σ(−2.2 + 0.9v + 1.2·shock)) (WP A.1)")
reg("hold_v", 0.9, "ASSUMED", "WP A.1")
reg("hold_shock", 1.2, "ASSUMED", "WP A.1")
# ---- ASSUMED: Phase 4 (Eq. 1–2) ------------------------------------------------------------------------
reg("tau_g", 60.0, "ASSUMED", "Low-grade cutoff on 0–100 scale (WP Table 4)")
reg("tau_a", 0.75, "ASSUMED", "Low-attendance cutoff (WP Table 4)")
reg("beta0", -9.0, "ASSUMED", "Eq. 1 intercept; chosen jointly with beta under illustrative submodel; NOT fitted to pilot population (WP Table 3/4)")
reg("beta", 6.0, "ASSUMED", "Eq. 1 shared weight beta1=beta2=beta3 (equal-weight symmetry assumption, WP §8)")
reg("design_sensitivity", 0.70, "ASSUMED", "Design target: share of genuinely high-vulnerability student-terms offered support (WP §8)")
reg("design_false_flag", 0.25, "ASSUMED", "Design target: share of supported student-terms with genuinely low vulnerability (WP §8)")
# ---- ASSUMED: Phase 5a (Eq. 3) -----------------------------------------------------------------------
reg("gamma1", 0.8, "ASSUMED", "Hazard slope on standardized vulnerability (WP A.1)")
reg("gamma2", 1.5, "ASSUMED", "Hazard slope on shock indicator (WP A.1)")
# ---- ASSUMED: validation thresholds ------------------------------------------------------------------
reg("heldout_refit_tol", 0.01, "ASSUMED", "Held-out refit: pooled attrition within 1 pp of 29.4% (WP §11)")
reg("heldout_band_lo", 1.2, "ASSUMED", "Held-out tier-ratio band, lower edge (WP §11)")
reg("heldout_band_hi", 1.7, "ASSUMED", "Held-out tier-ratio band, upper edge (WP §11)")
reg("recov_abs_window", 0.05, "ASSUMED", "Recoverability: point estimate within max(5 pp, 2·SD across runs) (WP Table 5)")
reg("edge_min_share", 0.01, "ASSUMED", "'Non-trivial' edge-case rate: >= 1% of the relevant event set. WP does not define 'non-trivial' (open flag OF-6)")
reg("edge_min_count", 100, "ASSUMED", "'Non-trivial' edge-case rate: >= 100 student-terms (OF-6)")
reg("targeting_high_current_quantile", 0.70, "ASSUMED", "Diagnostic only: top 30% of hazard linear predictor = 'genuinely high current vulnerability' (matches WP Table 3 illustrative 30% split)")
# ---- ASSUMED: non-core attributes (WP App. B: 'marginal forms unconstrained by this paper') ----------
reg("entry_year", 2018, "ASSUMED", "Single calibrated cohort: Class of 2022 = entered AY 2018–19. Constant by design (OF-3)")
reg("program_shares", {"Business & Management": 0.22, "Education": 0.14, "Engineering & IT": 0.24,
                       "Health Sciences": 0.12, "Arts, Sciences & Social Sci.": 0.20, "Other": 0.08},
    "ASSUMED", "Programme mix; causally inert in this version (affects course load only)")
reg("program_courses", {"Business & Management": 6.0, "Education": 6.0, "Engineering & IT": 6.5,
                        "Health Sciences": 7.0, "Arts, Sciences & Social Sci.": 6.0, "Other": 6.0},
    "ASSUMED", "Mean courses per term by programme; 3 units per course")
reg("course_noise_sd", 0.8, "ASSUMED", "SD of per-term course-count noise (counts clipped to 3–9)")
reg("pass_midpoint", 62.0, "ASSUMED", "P(pass course) = σ((grade − 62)/4)")
reg("pass_scale", 4.0, "ASSUMED", "see above")
reg("dist_median_km", 8.0, "ASSUMED", "distance_km = 8·exp(0.35·tier + 0.10·v + 0.8z)")
reg("dist_tier", 0.35, "ASSUMED", "High-tier students live further out on average")
reg("dist_v", 0.10, "ASSUMED", "Weak link between distance and latent vulnerability (permitted by App. B.2 leakage rule)")
reg("dist_sigma", 0.8, "ASSUMED", "Log-scale spread of distance")
reg("subsidy_by_bracket", {1: 0.45, 2: 0.35, 3: 0.20, 4: 0.10, 5: 0.05}, "ASSUMED", "P(subsidy at entry | income bracket 1=lowest)")
reg("subsidy_lapse_prob", 0.25, "ASSUMED", "Per-term P(permanent subsidy lapse) when the term grade < tau_g")
reg("overdue_median_php", 5000.0, "ASSUMED", "Median overdue amount (PHP) when a hold is active")
reg("overdue_v", 0.15, "ASSUMED", "Overdue amount rises mildly with vulnerability")
reg("overdue_sigma", 0.6, "ASSUMED", "Log-scale spread of overdue amount")

assert P("beta") > 0 and P("gamma1") > 0 and P("gamma2") > 0          # sign constraints in Eq. 1 and 3
param_log = pd.DataFrame([{"parameter": k, "value": json.dumps(v["value"]) if isinstance(v["value"], dict) else v["value"],
                           "tag": v["tag"], "basis": v["basis"]} for k, v in PARAM.items()])
assert param_log["tag"].isin(TAGS).all()
print(param_log["tag"].value_counts().to_string())
display(param_log[param_log.tag != "ASSUMED"][["parameter", "value", "tag", "basis"]].style.hide(axis="index"))

## 3 · Generator engine (Phases 1–5a)

**Causal structure** (whitepaper Fig. 2). Two latent quantities — baseline vulnerability $v_i$ and per-term shock $s_{i,t}$ — cause every recorded value. They are **never exported**.

$$
v_i \;\rightarrow\; (\text{grade}_{i,t},\ \text{attendance}_{i,t},\ \text{hold}_{i,t}) \;\rightarrow\; \text{risk}_{i,t+1} \;\rightarrow\; \text{intervention}_{i,t+1}
$$
$$
(v_i,\ s_{i,t},\ \text{intervention}_{i,t}) \;\rightarrow\; h_{i,t} \;\rightarrow\; \text{dropout}_{i,t}
$$

* **Eq. 1** $\ \text{risk}_{i,t}=\sigma\!\big(\beta_0+\beta_1\mathbb 1[\text{grade}_{i,t-1}<\tau_g]+\beta_2\mathbb 1[\text{att}_{i,t-1}<\tau_a]+\beta_3\,\text{hold}_{i,t-1}\big)$, defined for $t\ge2$ only.
* **Eq. 2** $\ \text{intervention}_{i,t}\sim\text{Bernoulli}(\text{risk}_{i,t})$; always 0 in term 1.
* **Eq. 3** $\ h_{i,t}=\sigma(\gamma_0+\gamma_1 v_i+\gamma_2 s_{i,t})\,(1-0.30\cdot\text{intervention}_{i,t})$.
* **Eq. 4** $\ \text{dropout}_{i,t}\sim\text{Bernoulli}(h_{i,t})$ for students still enrolled.

**Common random numbers (Law 2015; App. B.3 #2–3).** Every random quantity is drawn *once* from its own named stream (`SeedSequence.spawn`) as a full $N\times T$ array. Parameters then act only as deterministic transformations of those fixed draws. Two consequences: (i) changing one variable does not perturb any other stream; (ii) tier-level attrition is a **monotone** function of $(\delta,\gamma_0)$, which is what lets the calibration loop use bisection.

A student's *potential* observables are generated for all 8 terms, then truncated at the dropout term. This is valid because assignment depends only on prior-term observables, never on dropout history.

In [ ]:
sig = lambda x: 1.0 / (1.0 + np.exp(-x))

STREAMS = ["tier","work","inc","eps","shock","eg","ea","hold","assign","drop",
           "program","dist","subsidy","lapse","units","courses","overdue"]
MAX_COURSES = 9

def draw_noise(N, T, entropy, noncore=True):
    # All random draws, once, from independent named streams (common random numbers).
    ss = np.random.SeedSequence(entropy)
    rng = {n: np.random.default_rng(c) for n, c in zip(STREAMS, ss.spawn(len(STREAMS)))}
    d = dict(
        tier   = rng["tier"].random(N) < P("tier_share"),            # Phase 1
        work   = rng["work"].random(N) < P("work_share"),
        inc    = rng["inc"].standard_normal(N),
        eps    = rng["eps"].standard_normal(N),
        shock  = rng["shock"].random((N, T)) < P("shock_prob"),      # Phase 2 (term shock, independent of everything)
        eg     = rng["eg"].standard_normal((N, T)),                  # Phase 3 noise
        ea     = rng["ea"].standard_normal((N, T)),
        u_hold = rng["hold"].random((N, T)),
        u_assign = rng["assign"].random((N, T)),                     # Phase 4
        u_drop = rng["drop"].random((N, T)),                         # Phase 5a
    )
    if noncore:
        d.update(program_u = rng["program"].random(N), z_dist = rng["dist"].standard_normal(N),
                 u_subsidy = rng["subsidy"].random(N), u_lapse = rng["lapse"].random((N, T)),
                 z_units = rng["units"].standard_normal((N, T)),
                 u_course = rng["courses"].random((N, T, MAX_COURSES)),
                 z_overdue = rng["overdue"].standard_normal((N, T)))
    return d

def simulate_latent(d, delta):
    # Phases 2–4: baseline vulnerability, observables, risk score (Eq.1), intervention (Eq.2).
    N, T = d["shock"].shape
    v_raw = (delta * d["tier"] + P("vuln_work") * d["work"] + P("vuln_inc") * d["inc"] + P("vuln_eps") * d["eps"])
    v = (v_raw - v_raw.mean()) / v_raw.std()                          # standardized across the cohort
    vc, sh = v[:, None], d["shock"].astype(float)
    grade = P("grade_intercept") + P("grade_v")*vc + P("grade_shock")*sh + P("grade_noise")*d["eg"]
    att   = P("att_intercept")  + P("att_v")*vc    + P("att_shock")*sh   + P("att_noise")*d["ea"]
    hold  = d["u_hold"] < sig(P("hold_intercept") + P("hold_v")*vc + P("hold_shock")*sh)
    low_g, low_a = grade < P("tau_g"), att < P("tau_a")               # comparisons on UNclipped values
    b1 = b2 = b3 = P("beta")
    risk = np.full((N, T), np.nan)                                    # undefined in term 1
    risk[:, 1:] = sig(P("beta0") + b1*low_g[:, :-1] + b2*low_a[:, :-1] + b3*hold[:, :-1])
    interv = np.zeros((N, T), bool)                                   # no assignment in term 1
    interv[:, 1:] = d["u_assign"][:, 1:] < risk[:, 1:]
    cell = np.zeros((N, T), int)                                      # prior-term signal cell (4*lowG + 2*lowA + hold)
    cell[:, 1:] = 4*low_g[:, :-1] + 2*low_a[:, :-1] + hold[:, :-1]
    return dict(v=v, shock=d["shock"], grade=grade, att=att, hold=hold, low_g=low_g, low_a=low_a,
                risk=risk, interv=interv, cell=cell)

def apply_hazard(L, u_drop, gamma0):
    # Phase 5a: Eq. 3–4. Returns potential-dropout matrix, attrition flag, dropout/censor term, hazard.
    N, T = u_drop.shape
    lin = gamma0 + P("gamma1") * L["v"][:, None] + P("gamma2") * L["shock"]
    hz = sig(lin) * (1.0 - P("intervention_effect") * L["interv"])
    dp = u_drop < hz
    attr = dp.any(axis=1)
    first = np.where(attr, dp.argmax(axis=1) + 1, T)                  # T_i: dropout term, or last term if censored
    return dp, attr, first, hz

def bisect(f, lo, hi, target, n=45):
    # f must be non-decreasing; returns x with f(x) ~ target
    assert f(lo) <= target <= f(hi), "target not bracketed — widen the search interval"
    for _ in range(n):
        mid = 0.5 * (lo + hi)
        if f(mid) < target: lo = mid
        else: hi = mid
    return 0.5 * (lo + hi)
print("engine defined")

## 4 · Phase 5b — Calibration loop
Two tier targets (high 67.1 %, typical 45.9 %; both Class of 2022) and three hazard parameters ($\gamma_0,\gamma_1,\gamma_2$) are under-identified (whitepaper §9). Following Appendix A, $\gamma_1=0.8$ and $\gamma_2=1.5$ are fixed as `ASSUMED`, and the loop fits **$\gamma_0$ and the tier shift $\delta$** by nested bisection:

* inner: for a given $\delta$, find $\gamma_0$ so the **typical**-tier rate hits its target;
* outer: find $\delta$ so the **high**-tier rate hits its target.

⚠️ **This is calibration-until-convergence, not validation** (whitepaper §11): passing these two targets afterwards only shows the loop converged. Independent evidence comes from the held-out check and the Confounding/Recoverability checks in §7.

In [ ]:
N, T = CONFIG["N_STUDENTS"], P("T_MAX")
ENTROPY_MAIN = [CONFIG["MASTER_SEED"], 0]
t0 = time.time()
d_main = draw_noise(N, T, ENTROPY_MAIN)

def calibrate(d, verbose=True):
    tier = d["tier"]
    def inner(delta):
        L = simulate_latent(d, delta)
        g0 = bisect(lambda g: apply_hazard(L, d["u_drop"], g)[1][~tier].mean(), -10, 5, P("target_typical_2022"))
        high = apply_hazard(L, d["u_drop"], g0)[1][tier].mean()
        return g0, high
    delta = bisect(lambda dl: inner(dl)[1], 0.0, 4.0, P("target_high_2022"), n=28)
    g0, high = inner(delta)
    return delta, g0

if CONFIG["RECALIBRATE"]:
    delta_hat, gamma0_hat = calibrate(d_main)
    print(f"Calibrated in {time.time()-t0:.1f}s:  delta = {delta_hat:.4f}   gamma0 = {gamma0_hat:.4f}")
else:
    delta_hat, gamma0_hat = CONFIG["FROZEN_CALIBRATION"]["delta"], CONFIG["FROZEN_CALIBRATION"]["gamma0"]
    print(f"Using FROZEN calibration: delta = {delta_hat}, gamma0 = {gamma0_hat}")
set_calibrated("delta", delta_hat); set_calibrated("gamma0", gamma0_hat)
param_log.loc[param_log.parameter == "delta", "value"] = delta_hat
param_log.loc[param_log.parameter == "gamma0", "value"] = gamma0_hat

L_main = simulate_latent(d_main, P("delta"))
dp_main, attr_main, first_main, hz_main = apply_hazard(L_main, d_main["u_drop"], P("gamma0"))
rate_hi, rate_ty, rate_all = attr_main[d_main["tier"]].mean(), attr_main[~d_main["tier"]].mean(), attr_main.mean()
print(f"High tier  {rate_hi:6.1%}  (target {P('target_high_2022'):.1%})")
print(f"Typical    {rate_ty:6.1%}  (target {P('target_typical_2022'):.1%})")
print(f"Pooled     {rate_all:6.1%}  (derived blend {P('tier_share')*P('target_high_2022')+(1-P('tier_share'))*P('target_typical_2022'):.1%})")

## 5 · Assemble the eight tables and export
Schema follows Appendix B (Table 8). Six tables are student-term panels (`student_id, term`); `Regions` and `Students` are fixed. Rows exist only for terms $1,\dots,T_i$.

**Never exported** (App. B.2): baseline vulnerability, shock indicators, the hazard $h_{i,t}$, any calibration intermediate. These are kept in an in-memory `AUDIT` object used *only* for the audits in §7–8.

**Non-core attributes** (programme, distance, unit counts, overdue amount, subsidy fields) have unconstrained marginal forms in the whitepaper; the forms used here are `ASSUMED` (§2) and respect the **leakage rule**: they depend only on entry attributes, the latent state and *current-term* observables — never on any dropout outcome or later-term value. Programme is causally inert in this version.

**Responsible-use design (whitepaper §13.3):** the exported tables carry only `region_tier` (`high`/`typical`) — never a region name — and no free-text or narrative fields.

In [ ]:
def build_tables(d, L, gamma0):
    N, T = L["grade"].shape
    dp, attr, first, hz = apply_hazard(L, d["u_drop"], gamma0)
    tix = np.arange(1, T + 1)
    mask = tix[None, :] <= first[:, None]                 # rows exist for terms 1..T_i only
    r, c = np.nonzero(mask)                               # row-major => ordered by student then term
    sid, term = r + 1, c + 1
    tier = d["tier"]

    # ---- Students (fixed) -------------------------------------------------------------------------
    income_bracket = pd.qcut(d["inc"], 5, labels=False).astype(int) + 1     # 1 = lowest income
    names = list(P("program_shares")); cum = np.cumsum(list(P("program_shares").values()))
    prog = np.array(names)[np.minimum(np.searchsorted(cum, d["program_u"], side="right"), len(names) - 1)]
    distance = np.exp(np.log(P("dist_median_km")) + P("dist_tier")*tier + P("dist_v")*L["v"] + P("dist_sigma")*d["z_dist"])
    p_sub = np.array([P("subsidy_by_bracket")[b] for b in range(1, 6)])[income_bracket - 1]
    sub0 = d["u_subsidy"] < p_sub
    Students = pd.DataFrame({"student_id": np.arange(1, N + 1),
        "region_tier": np.where(tier, "high", "typical"), "entry_year": P("entry_year"), "program": prog,
        "income_bracket": income_bracket, "working_student": d["work"],
        "distance_km": distance.round(2), "subsidy_at_entry": sub0})
    Regions = pd.DataFrame({"region_tier": ["high", "typical"], "tier_share": [P("tier_share"), 1 - P("tier_share")],
        "target_attrition": [P("target_high_2022"), P("target_typical_2022")],
        "target_source": ["CPBRD 2025 (CHED FY2026 Agency Budget Notes), Class-of-2022 column: mean of BARMM, VII, IX, CAR; EDCOM II 2025",
                          "CPBRD 2025, Class-of-2022 column: enrollment-weighted mean of the 13 remaining regions"]})

    # ---- panel observables --------------------------------------------------------------------------
    grade = np.clip(L["grade"][mask], 0, 100); att = np.clip(L["att"][mask], 0, 1); hold = L["hold"][mask]
    course_mean = pd.Series(prog).map(P("program_courses")).to_numpy()[r]
    n_course = np.clip(np.rint(course_mean + P("course_noise_sd") * d["z_units"][mask]), 3, MAX_COURSES).astype(int)
    p_pass = sig((grade - P("pass_midpoint")) / P("pass_scale"))
    passed = ((d["u_course"][mask] < p_pass[:, None]) & (np.arange(MAX_COURSES)[None, :] < n_course[:, None])).sum(1)
    lapse_ev = (L["grade"] < P("tau_g")) & (d["u_lapse"] < P("subsidy_lapse_prob"))
    lapsed = np.maximum.accumulate(lapse_ev, axis=1)                           # lapse is permanent
    sub_status = (sub0[:, None] & ~lapsed)[mask]
    overdue = np.where(hold, np.exp(np.log(P("overdue_median_php")) + P("overdue_v")*L["v"][r]
                                    + P("overdue_sigma")*d["z_overdue"][mask]), 0.0)
    drop_flag = (term == first[r]) & attr[r]

    key = {"student_id": sid, "term": term}
    Enrollments = pd.DataFrame({**key, "enrollment_status": np.where(drop_flag, "withdrawn", "continuing"),
                                "censored": (term == T) & ~attr[r]})
    CourseGrades = pd.DataFrame({**key, "term_avg_score": grade.round(4), "units_attempted": 3 * n_course,
                                 "units_passed": 3 * passed})
    AttendanceRecords = pd.DataFrame({**key, "attendance_rate": att.round(6)})
    FinancialHolds = pd.DataFrame({**key, "hold_flag": hold, "overdue_amount": overdue.round(2), "subsidy_status": sub_status})
    SupportInterventions = pd.DataFrame({**key, "risk_score": L["risk"][mask], "intervention": L["interv"][mask]})
    PersistenceOutcomes = pd.DataFrame({**key, "dropout_flag": drop_flag, "attrition_flag": attr[r]})

    tables = {"Regions": Regions, "Students": Students, "Enrollments": Enrollments, "CourseGrades": CourseGrades,
              "AttendanceRecords": AttendanceRecords, "FinancialHolds": FinancialHolds,
              "SupportInterventions": SupportInterventions, "PersistenceOutcomes": PersistenceOutcomes}
    audit = {"student": pd.DataFrame({"student_id": np.arange(1, N + 1), "baseline_vulnerability": L["v"]}),
             "panel": pd.DataFrame({**key, "shock": L["shock"][mask], "hazard": hz[mask]})}
    return tables, audit

TABLES, AUDIT = build_tables(d_main, L_main, P("gamma0"))
for k, v in TABLES.items(): print(f"{k:22s} {len(v):>9,} rows × {v.shape[1]} cols")

In [ ]:
def sha256_df(df): return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()

def export_all(tables, audit, outdir):
    hashes = {}
    for name, df in tables.items():
        if CONFIG["EXPORT_CSV"]:
            df.to_csv(outdir / f"{name}.csv", index=False)
        hashes[name] = sha256_df(df)
    param_log.to_csv(outdir / "parameter_log.csv", index=False)
    meta = {"notice": SYNTHETIC_BANNER, "spec_version": SPEC_VERSION, "generator_version": GENERATOR_VERSION,
            "master_seed": CONFIG["MASTER_SEED"], "n_students": int(len(tables["Students"])), "recalibrated": CONFIG["RECALIBRATE"],
            "calibrated": {"delta": P("delta"), "gamma0": P("gamma0")},
            "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                         "scipy": scipy.__version__, "statsmodels": statsmodels.__version__, "platform": platform.platform()},
            "rows": {k: int(len(v)) for k, v in tables.items()}, "sha256": hashes,
            "reproducibility": "Bitwise reproducible only on the same platform and library versions; across platforms equivalence is statistical (WP App. B.3 #5)."}
    (outdir / "metadata.json").write_text(json.dumps(meta, indent=2))
    (outdir / "DATASET_NOTICE.txt").write_text(SYNTHETIC_BANNER + "\nThe latent state (baseline vulnerability, shocks, hazard) is not included.\n")
    if CONFIG["WRITE_AUDIT_LATENT"]:
        ad = outdir / "_PRIVATE_AUDIT_DO_NOT_SHARE"; ad.mkdir(exist_ok=True)
        for k, v in audit.items(): v.to_csv(ad / f"audit_{k}.csv", index=False)
    return meta

META = export_all(TABLES, AUDIT, OUT)
print("Exported to", OUT.resolve(), "→", sorted(p.name for p in OUT.iterdir()))
# a latent column must never appear in an exported table
LATENT_NAMES = {"baseline_vulnerability", "shock", "hazard", "vulnerability"}
assert not any(LATENT_NAMES & set(df.columns) for df in TABLES.values()), "latent leaked into export!"
print("✔ no latent column in any exported table")

In [ ]:
# One illustrative individual — drawn from the TYPICAL tier, per WP §13.3(iv); tabular fields only, no narrative.
sid_ex = int(TABLES["Students"].query("region_tier == 'typical'").merge(
    TABLES["PersistenceOutcomes"].query("dropout_flag")[["student_id"]], on="student_id").student_id.iloc[0])
ex = TABLES["Enrollments"].query("student_id == @sid_ex")
for nm in ["CourseGrades","AttendanceRecords","FinancialHolds","SupportInterventions","PersistenceOutcomes"]:
    ex = ex.merge(TABLES[nm], on=["student_id","term"])
show(ex, n=8)
show(TABLES["Students"].query("student_id == @sid_ex"))

## 6 · Sanity checks
*Realistic ranges, sensible correlations, keys and integrity rules* (the official "definition of done" and App. B.2). Section 7's Table-5 scorecard follows.

In [ ]:
S = TABLES
checks = []
def chk(name, ok, detail=""): checks.append({"check": name, "pass": bool(ok), "detail": detail})

# keys / referential integrity
for nm in ["Enrollments","CourseGrades","AttendanceRecords","FinancialHolds","SupportInterventions","PersistenceOutcomes"]:
    chk(f"{nm}: unique (student_id, term)", not S[nm].duplicated(["student_id","term"]).any())
    chk(f"{nm}: all student_id in Students", S[nm].student_id.isin(S["Students"].student_id).all())
chk("Students: unique student_id", S["Students"].student_id.is_unique)
chk("Students.region_tier ⊂ Regions", S["Students"].region_tier.isin(S["Regions"].region_tier).all())
rows = [len(S[n]) for n in ["Enrollments","CourseGrades","AttendanceRecords","FinancialHolds","SupportInterventions","PersistenceOutcomes"]]
chk("six panel tables have identical row counts", len(set(rows)) == 1, str(rows[0]))

# no row after the dropout term; censoring semantics
last = S["Enrollments"].groupby("student_id").term.max()
wd = S["Enrollments"].query("enrollment_status == 'withdrawn'").set_index("student_id").term
chk("withdrawn row is each withdrawing student's LAST row", (last.loc[wd.index] == wd).all())
chk("each student withdraws at most once", S["Enrollments"].query("enrollment_status=='withdrawn'").student_id.is_unique)
cens = S["Enrollments"].query("censored")
chk("censored only on term T_MAX, never for a withdrawer", (cens.term == T).all() and not cens.student_id.isin(wd.index).any())
chk("students never reaching a dropout: have all T_MAX rows", (last[~last.index.isin(wd.index)] == T).all())
chk("attrition_flag == (student ever withdrew)", S["PersistenceOutcomes"].groupby("student_id").attrition_flag.first().eq(last.index.isin(wd.index)).all())

# term-1 rule (Section 8)
si = S["SupportInterventions"]
chk("risk_score null in term 1 only", si[si.term == 1].risk_score.isna().all() and si[si.term > 1].risk_score.notna().all())
chk("intervention always False in term 1", not si[si.term == 1].intervention.any())
chk("risk_score ∈ (0,1) for t≥2", si[si.term > 1].risk_score.between(0, 1, inclusive="neither").all())

# ranges
cg, ar, fh = S["CourseGrades"], S["AttendanceRecords"], S["FinancialHolds"]
chk("term_avg_score ∈ [0,100]", cg.term_avg_score.between(0, 100).all(), f"{cg.term_avg_score.min():.1f}–{cg.term_avg_score.max():.1f}")
chk("attendance_rate ∈ [0,1]", ar.attendance_rate.between(0, 1).all())
chk("units_passed ≤ units_attempted", (cg.units_passed <= cg.units_attempted).all())
chk("overdue_amount == 0 iff no hold", ((fh.overdue_amount == 0) == ~fh.hold_flag).all())
chk("distance_km ≥ 0 and income_bracket ∈ 1..5", (S["Students"].distance_km >= 0).all() and S["Students"].income_bracket.between(1, 5).all())
chk("no nulls outside risk_score term 1", S["Students"].notna().all().all() and cg.notna().all().all() and ar.notna().all().all() and fh.notna().all().all())
sc = pd.DataFrame(checks); print(f"{sc['pass'].sum()}/{len(sc)} integrity checks pass"); display(sc[~sc["pass"]] if (~sc["pass"]).any() else sc.tail(3))
assert sc["pass"].all(), "integrity failure"

In [ ]:
# Section 7 of the WP: observables are INDEPENDENT given the latent state but CORRELATED unconditionally.
# Test it directly using the private audit object (analysts never see this).
pan = (S["CourseGrades"][["student_id","term","term_avg_score"]]
       .merge(S["AttendanceRecords"][["student_id","term","attendance_rate"]]).merge(S["FinancialHolds"][["student_id","term","hold_flag"]])
       .merge(AUDIT["panel"][["student_id","term","shock"]]).merge(AUDIT["student"]))
def resid(y, X):
    X = sm.add_constant(X.astype(float)); return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
Xl = pan[["baseline_vulnerability","shock"]].to_numpy()
cols = {"grade": pan.term_avg_score.to_numpy(float), "attendance": pan.attendance_rate.to_numpy(float), "hold": pan.hold_flag.to_numpy(float)}
unc = pd.DataFrame(cols).corr()
con = pd.DataFrame({k: resid(v, Xl) for k, v in cols.items()}).corr()
print("Unconditional correlations (what an analyst sees):"); display(unc.round(3))
print("Correlations after removing the latent state (hidden):"); display(con.round(3))
off = lambda m: m.values[np.triu_indices(3, 1)]
assert np.abs(off(unc)).min() > 0.10 and np.abs(off(con)).max() < 0.05
print("✔ correlated unconditionally, ~independent given (v, shock) — as WP §7 requires")

## 7 · Validation scorecard (whitepaper Table 5)

### 7.1 Estimand and estimators
The true effect is a **risk ratio** (RR = 0.70, i.e. 30 % relative hazard reduction) — *not* an odds ratio, which drifts away from 0.70 as baseline hazard rises (WP §11). Estimators are therefore scored on the RR.

All analyses below are built from the **exported tables only** — the position of a real Student Success Analyst. The prior-term signals are reconstructed from `CourseGrades`, `AttendanceRecords` and `FinancialHolds`, then checked against the internal arrays to prove the export contains what an analyst needs.

* *Naive*: ratio of raw dropout rates, treated vs untreated (student-terms $t\ge2$), student-level bootstrap CI.
* *Mantel–Haenszel RR* across the 8 prior-signal cells (Greenland–Robins variance) and *Poisson regression with cluster-robust SEs* — both borrow strength across cells.
* *IPW with the true propensity* (four variants: Hajek-normalised vs Horvitz–Thompson, each untrimmed and with propensities clipped to [0.05, 0.95]) and the *odds ratio from a cell-adjusted logistic regression* — shown as **diagnostics** of the positivity problem and the estimand mismatch (WP §8, §11). Do not read a single IPW number as 'recovered': the point of showing four is that the answer **depends on implementation details** when two of four cells are near-deterministic.

In [ ]:
def analyst_frame(tb):
    df = (tb["SupportInterventions"]
          .merge(tb["CourseGrades"][["student_id","term","term_avg_score"]], on=["student_id","term"])
          .merge(tb["AttendanceRecords"][["student_id","term","attendance_rate"]], on=["student_id","term"])
          .merge(tb["FinancialHolds"][["student_id","term","hold_flag"]], on=["student_id","term"])
          .merge(tb["PersistenceOutcomes"][["student_id","term","dropout_flag"]], on=["student_id","term"])
          .sort_values(["student_id","term"]).reset_index(drop=True))
    g = df.groupby("student_id")
    lg, la, lh = g["term_avg_score"].shift(1), g["attendance_rate"].shift(1), g["hold_flag"].shift(1)
    df["cell"] = (4*(lg < P("tau_g")) + 2*(la < P("tau_a")) + lh.astype(float)).where(df.term >= 2)
    return df[df.term >= 2].assign(cell=lambda x: x.cell.astype(int))

AF = analyst_frame(TABLES)
y  = AF.dropout_flag.to_numpy(bool); Tr = AF.intervention.to_numpy(bool); cell = AF.cell.to_numpy(int)
sid = AF.student_id.to_numpy(); prop = AF.risk_score.to_numpy(float)

# verification: reconstructed-from-export == internal arrays
tix = np.arange(1, T + 1); m_all = tix[None, :] <= first_main[:, None]; m2 = m_all & (tix[None, :] >= 2)
assert len(AF) == m2.sum() and (cell == L_main["cell"][m2]).all() and (Tr == L_main["interv"][m2]).all() and (y == (dp_main & m_all)[m2]).all()
print(f"✔ analyst-view reconstruction matches the engine exactly on {len(AF):,} student-terms with t ≥ 2")

def mh_rr(y, Tr, cell):
    # Mantel–Haenszel risk ratio (treated / control) and Greenland–Robins 95% CI on log RR
    K = 8
    a  = np.bincount(cell, weights=(y & Tr),  minlength=K); n1 = np.bincount(cell, weights=Tr.astype(float),  minlength=K)
    c_ = np.bincount(cell, weights=(y & ~Tr), minlength=K); n0 = np.bincount(cell, weights=(~Tr).astype(float), minlength=K)
    n = n1 + n0; ok = n > 0
    R = (a[ok]*n0[ok]/n[ok]).sum(); S_ = (c_[ok]*n1[ok]/n[ok]).sum(); rr = R / S_
    var = ((n1[ok]*n0[ok]*(a[ok]+c_[ok]) - a[ok]*c_[ok]*n[ok]) / n[ok]**2).sum() / (R * S_)
    se = np.sqrt(var); return rr, (rr*np.exp(-1.96*se), rr*np.exp(1.96*se))

def poisson_rr(y, Tr, cell, groups):
    X = pd.get_dummies(pd.Series(cell), prefix="c", drop_first=True, dtype=float)
    X.insert(0, "T", Tr.astype(float)); X = sm.add_constant(X, has_constant="add")
    g = pd.factorize(groups)[0]
    m = sm.GLM(y.astype(float), X, family=sm.families.Poisson()).fit(cov_type="cluster", cov_kwds={"groups": g})
    b, se = m.params["T"], m.bse["T"]; return np.exp(b), (np.exp(b - 1.96*se), np.exp(b + 1.96*se))

def logit_or(y, Tr, cell):
    X = pd.get_dummies(pd.Series(cell), prefix="c", drop_first=True, dtype=float)
    X.insert(0, "T", Tr.astype(float)); X = sm.add_constant(X, has_constant="add")
    return float(np.exp(sm.Logit(y.astype(float), X).fit(disp=0).params["T"]))

def ipw_rr(y, Tr, p, clip=None, hajek=True):
    # IPW risk ratio with the TRUE propensity (exported as risk_score). hajek=True normalises weights; False = Horvitz–Thompson.
    p = np.clip(p, *clip) if clip else p
    w = np.where(Tr, 1/p, 1/(1-p))
    if hajek: m1 = (w*Tr*y).sum() / (w*Tr).sum();  m0 = (w*(~Tr)*y).sum() / (w*(~Tr)).sum()
    else:     m1 = (Tr*y/p).mean();                m0 = ((~Tr)*y/(1-p)).mean()
    return m1 / m0, w.max()

def boot_naive(y, Tr, sid, B, seed):
    u, inv = np.unique(sid, return_inverse=True); M = len(u)
    E1 = np.bincount(inv, weights=(y & Tr), minlength=M);  N1 = np.bincount(inv, weights=Tr.astype(float), minlength=M)
    E0 = np.bincount(inv, weights=(y & ~Tr), minlength=M); N0 = np.bincount(inv, weights=(~Tr).astype(float), minlength=M)
    rng = np.random.default_rng(seed); out = np.empty(B)
    for b in range(B):
        s = rng.integers(0, M, M); out[b] = 1 - (E1[s].sum()/N1[s].sum()) / (E0[s].sum()/N0[s].sum())
    return np.percentile(out, [2.5, 97.5])

rel = lambda rr: 1 - rr                       # relative reduction = 1 − RR
naive_rr = y[Tr].mean() / y[~Tr].mean()
naive_ci = boot_naive(y, Tr, sid, CONFIG["N_BOOTSTRAP"], CONFIG["MASTER_SEED"] + 7)
mh, mh_ci = mh_rr(y, Tr, cell)
po, po_ci = poisson_rr(y, Tr, cell, sid)
ipw_hu, wmax = ipw_rr(y, Tr, prop); ipw_hc, _ = ipw_rr(y, Tr, prop, (0.05, 0.95))
ipw_tu, _ = ipw_rr(y, Tr, prop, hajek=False); ipw_tc, _ = ipw_rr(y, Tr, prop, (0.05, 0.95), hajek=False)
OR = logit_or(y, Tr, cell)

est = pd.DataFrame([
  ("Naive (unadjusted)",                      rel(naive_rr), naive_ci[0], naive_ci[1], "bootstrap by student"),
  ("IPW Hajek, untrimmed",                    rel(ipw_hu), np.nan, np.nan, f"max weight {wmax:,.0f}; driven by a handful of terms"),
  ("IPW Hajek, clipped [.05,.95]",            rel(ipw_hc), np.nan, np.nan, "extrapolation into near-empty cells"),
  ("IPW Horvitz–Thompson, untrimmed",         rel(ipw_tu), np.nan, np.nan, "same weights, different normalisation"),
  ("IPW Horvitz–Thompson, clipped [.05,.95]", rel(ipw_tc), np.nan, np.nan, "same weights, different normalisation"),
  ("Logistic odds ratio (cell-adjusted)",     rel(OR), np.nan, np.nan, "wrong estimand — OR is not the RR"),
  ("Mantel–Haenszel RR (8 cells)",            rel(mh), rel(mh_ci[1]), rel(mh_ci[0]), "Greenland–Robins CI (ignores clustering)"),
  ("Poisson, cluster-robust (8 cells)",       rel(po), rel(po_ci[1]), rel(po_ci[0]), "cluster = student"),
], columns=["estimator", "est. relative reduction", "CI lo", "CI hi", "note"])
TRUE_EFFECT = P("intervention_effect")
print(f"TRUE effect (LOCKED, by construction) = {TRUE_EFFECT:.0%}")
display(est.style.format({"est. relative reduction": "{:.1%}", "CI lo": "{:.1%}", "CI hi": "{:.1%}"}, na_rep="—").hide(axis="index"))
ipw_all = [rel(x) for x in (ipw_hu, ipw_hc, ipw_tu, ipw_tc)]
print(f"IPW spread across 4 variants on identical data: {min(ipw_all):.1%} to {max(ipw_all):.1%}  → IPW is not a dependable estimator here, "
      "even when an untrimmed variant happens to land near 30%.")

### 7.2 Positivity: the assignment cells
The targeting figures force a near-step assignment (whitepaper §8, Table 3): with $\beta_0=-9,\ \beta=6$ the risk is ≈0 for $k\le1$ and ≈1 for $k\ge2$ signals. Only the $k=1$ and $k=2$ cells contain both arms — the practical reason IPW is unstable and full standardisation extrapolates into near-empty cells.

In [ ]:
_c = AF.cell.to_numpy(int)
kcnt = ((_c >> 2) & 1) + ((_c >> 1) & 1) + (_c & 1)
pos = (AF.assign(k=kcnt).groupby("k").agg(student_terms=("intervention", "size"), treated=("intervention", "sum"))
         .assign(share_treated=lambda x: x.treated / x.student_terms, untreated=lambda x: x.student_terms - x.treated))
display(pos.style.format({"student_terms": "{:,}", "treated": "{:,}", "untreated": "{:,}", "share_treated": "{:.2%}"}))
print(f"Cells containing BOTH arms with ≥100 of each: {[int(k) for k in pos.index[(pos.treated >= 100) & (pos.untreated >= 100)]]}")

### 7.3 Replications and the held-out check
* **Replications (Law 2015):** `N_REPLICATES` independent populations at the *calibrated* parameters. They give (i) the estimator's SD that sets the Recoverability window, (ii) the stability of the tier rates, (iii) the **distribution** of the held-out tier ratio.
* **Held-out cohort (Class of 2023):** freeze everything except the overall hazard level, refit $\gamma_0$ so pooled attrition is within 1 pp of 29.4 %, then — *without further tuning* — require the high/typical ratio to fall in [1.2, 1.7] (observed 1.37). **Limits (WP §11):** low power (one structural number, wide band) and *not blind* — Class-of-2023 values were inspected while designing the check. A single-seed verdict is fragile here because the ratio's sampling SD is a few hundredths and the band's upper edge is close to the model's typical value; the replicate distribution is the honest summary.

In [ ]:
def heldout_ratio(L, d):
    tier = d["tier"]
    g1 = bisect(lambda g: apply_hazard(L, d["u_drop"], g)[1].mean(), -10, 5, P("national_2023_level"))
    a1 = apply_hazard(L, d["u_drop"], g1)[1]
    return a1[tier].mean() / a1[~tier].mean(), g1, a1[tier].mean(), a1[~tier].mean(), a1.mean()

def analysis_arrays(L, dp, first):
    m_all = tix[None, :] <= first[:, None]; m2 = m_all & (tix[None, :] >= 2)
    return (dp & m_all)[m2], L["interv"][m2], L["cell"][m2]

t0 = time.time(); reps = []
for r_ in range(1, CONFIG["N_REPLICATES"] + 1):
    d_r = draw_noise(N, T, [CONFIG["MASTER_SEED"], r_], noncore=False)
    L_r = simulate_latent(d_r, P("delta"))
    dp_r, at_r, fi_r, _ = apply_hazard(L_r, d_r["u_drop"], P("gamma0"))
    y_r, T_r, c_r = analysis_arrays(L_r, dp_r, fi_r)
    rr_r, _ = mh_rr(y_r, T_r, c_r)
    ho = heldout_ratio(L_r, d_r)
    reps.append({"rep": r_, "high": at_r[d_r["tier"]].mean(), "typical": at_r[~d_r["tier"]].mean(), "pooled": at_r.mean(),
                 "mh_reduction": 1 - rr_r, "naive_reduction": 1 - y_r[T_r].mean()/y_r[~T_r].mean(),
                 "heldout_ratio": ho[0], "heldout_gamma0": ho[1]})
REP = pd.DataFrame(reps)
print(f"{len(REP)} independent populations in {time.time()-t0:.0f}s")
display(REP[["high","typical","pooled","mh_reduction","heldout_ratio"]].agg(["mean","std","min","max"]).T
        .style.format("{:.4f}"))

In [ ]:
# ============================== TABLE 5 SCORECARD ==============================
tol = P("tol_pp")
blend = P("tier_share")*P("target_high_2022") + (1 - P("tier_share"))*P("target_typical_2022")
score = []
def add(measure, measured, benchmark, ok): score.append({"measure": measure, "measured": measured, "benchmark": benchmark, "verdict": "PASS" if ok else "FAIL"})

# 1. national cohort attrition (derived)
add("National attrition (derived blend)", f"{rate_all:.1%}", f"{blend:.1%} ± {tol:.0%}", abs(rate_all - blend) <= tol)
# 2. tier-level
add("Tier attrition — high",    f"{rate_hi:.1%}", f"{P('target_high_2022'):.1%} ± {tol:.0%}",    abs(rate_hi - P("target_high_2022")) <= tol)
add("Tier attrition — typical", f"{rate_ty:.1%}", f"{P('target_typical_2022'):.1%} ± {tol:.0%}", abs(rate_ty - P("target_typical_2022")) <= tol)
# 3. held-out (single run on the main population, then replicate distribution)
ho_ratio, ho_g, ho_h, ho_t, ho_p = heldout_ratio(L_main, d_main)
lo_, hi_ = P("heldout_band_lo"), P("heldout_band_hi")
ho_ok = (abs(ho_p - P("national_2023_level")) <= P("heldout_refit_tol")) and (lo_ <= ho_ratio <= hi_)
frac_in = ((REP.heldout_ratio >= lo_) & (REP.heldout_ratio <= hi_)).mean()
add("Held-out Class of 2023 — tier ratio (main run)", f"{ho_ratio:.3f} (γ0 {P('gamma0'):.2f}→{ho_g:.2f}; high {ho_h:.1%}, typical {ho_t:.1%})", f"in [{lo_}, {hi_}]; observed {P('observed_ratio_2023')}", ho_ok)
if ho_ok and frac_in < 0.8: score[-1]["verdict"] = "PASS (fragile)"      # single-seed pass that most replicate populations do not share
score.append({"measure": "Held-out — replicate distribution (diagnostic)", "measured": f"{REP.heldout_ratio.mean():.3f} ± {REP.heldout_ratio.std():.3f}; {frac_in:.0%} of {len(REP)} runs in band",
              "benchmark": "informational (WP pilot: 1.71 ± 0.04, 12% in band)", "verdict": "INFO"})
# 4. confounding presence
conf_ok = not (naive_ci[0] <= TRUE_EFFECT <= naive_ci[1])
add("Confounding presence", f"naive {rel(naive_rr):.1%} (95% CI {naive_ci[0]:.1%} to {naive_ci[1]:.1%})", f"CI excludes true {TRUE_EFFECT:.0%}", conf_ok)
# 5. recoverability: CI includes truth AND point estimate within max(5pp, 2·SD across replicate runs)
sd_runs = REP.mh_reduction.std(ddof=1); window = max(P("recov_abs_window"), 2*sd_runs)
for nm, pt, ci in [("Mantel–Haenszel", rel(mh), (rel(mh_ci[1]), rel(mh_ci[0]))), ("Poisson (cluster-robust)", rel(po), (rel(po_ci[1]), rel(po_ci[0])))]:
    ok = (ci[0] <= TRUE_EFFECT <= ci[1]) and abs(pt - TRUE_EFFECT) <= window
    add(f"Recoverability — {nm}", f"{pt:.1%} (95% CI {ci[0]:.1%} to {ci[1]:.1%})", f"CI ∋ {TRUE_EFFECT:.0%}; |err| ≤ {window:.1%} (=max(5pp, 2×SD {sd_runs:.1%}))", ok)
# 6. edge-case presence → filled in §8
# 7. parameter traceability
add("Parameter traceability", f"{len(PARAM)} parameters; tags: " + ", ".join(f"{k}={v}" for k, v in param_log.tag.value_counts().items()), "every parameter has exactly one tag", all(v["tag"] in TAGS for v in PARAM.values()))
SCORE_CORE = pd.DataFrame(score); display(SCORE_CORE.style.hide(axis="index"))
if window > 0.10:
    print(f"CAUTION: the Recoverability window is ±{window:.0%} (2×SD across runs). At this N the check has little power — a PASS is weakly informative.")
print(f"\nHeld-out ratio across {len(REP)} independent populations: {REP.heldout_ratio.mean():.3f} ± {REP.heldout_ratio.std():.3f}; "
      f"{frac_in:.0%} of runs inside [{lo_}, {hi_}].")

In [ ]:
# Realized targeting vs the Section-8 design targets (diagnostic, not a Table-5 pass/fail).
# 'Genuinely high current vulnerability' has no operational definition in the WP; here: top-q of the hazard linear predictor
# among enrolled student-terms t>=2. q=0.70 (top 30%) mirrors the WP's illustrative 30% split in Table 3 (ASSUMED);
# other cut-offs are shown so the conclusion does not hinge on that choice.
lin = (P("gamma1") * L_main["v"][:, None] + P("gamma2") * L_main["shock"])[m2]
rows_t = []
for q in (0.60, 0.70, 0.80, 0.90):
    hi_cur = lin >= np.quantile(lin, q)
    rows_t.append({"'high current vuln.' = top": f"{1-q:.0%} of student-terms", "realized sensitivity": Tr[hi_cur].mean(),
                   "realized false-flag share": (~hi_cur[Tr]).mean(),
                   "design targets": f"{P('design_sensitivity'):.0%} / {P('design_false_flag'):.0%}"})
display(pd.DataFrame(rows_t).style.format({"realized sensitivity": "{:.1%}", "realized false-flag share": "{:.1%}"}).hide(axis="index"))
share_sup = Tr.mean()
print(f"Share of student-terms receiving support: {share_sup:.1%}")
print(f"Upper bound on sensitivity = supported share / prevalence of 'high' terms: at 30% prevalence ≤ {share_sup/0.30:.1%}; "
      f"hitting {P('design_sensitivity'):.0%} would need prevalence ≤ {share_sup/P('design_sensitivity'):.1%} even with zero false flags.")

### 7.4 Is re-fitting the assignment parameters to the 70 % / 25 % targets well-posed?
Open flag OF-1 asked whether to re-fit $(\beta_0,\beta)$. This scan answers it on the **same population** ($\gamma_0,\delta$ held fixed, so tier attrition shifts slightly and is not re-calibrated — the scan is about targeting and overlap only). "High current vulnerability" = top 30 % of the hazard linear predictor (ASSUMED, as above).

In [ ]:
def targeting_scan(scenarios):
    out, saved = [], (PARAM["beta0"]["value"], PARAM["beta"]["value"])
    try:
        for b0, b in scenarios:
            PARAM["beta0"]["value"], PARAM["beta"]["value"] = b0, b          # temporary override; always restored below
            L_s = simulate_latent(d_main, P("delta")); _, _, fi_s, _ = apply_hazard(L_s, d_main["u_drop"], P("gamma0"))
            m2_s = (tix[None, :] <= fi_s[:, None]) & (tix[None, :] >= 2)
            Tr_s, c_s = L_s["interv"][m2_s], L_s["cell"][m2_s]
            lin_s = (P("gamma1") * L_s["v"][:, None] + P("gamma2") * L_s["shock"])[m2_s]
            hi_s = lin_s >= np.quantile(lin_s, P("targeting_high_current_quantile"))
            k_s = ((c_s >> 2) & 1) + ((c_s >> 1) & 1) + (c_s & 1)
            both = [c for c in range(4) if (Tr_s & (k_s == c)).sum() >= 100 and ((~Tr_s) & (k_s == c)).sum() >= 100]
            out.append({"β0": b0, "β": b, "risk(k=0..3)": " ".join(f"{sig(b0 + b*c):.2f}" for c in range(4)),
                        "supported share": Tr_s.mean(), "sensitivity": Tr_s[hi_s].mean(), "false-flag share": (~hi_s[Tr_s]).mean(),
                        "cells with both arms (≥100 each)": str(both)})
    finally:
        PARAM["beta0"]["value"], PARAM["beta"]["value"] = saved
    return pd.DataFrame(out)

SCAN = targeting_scan([(-9, 6), (-7, 6), (-5, 6), (-5, 3), (-3, 2), (-2, 1), (0, 2), (0, 6)])
display(SCAN.style.format({"supported share": "{:.1%}", "sensitivity": "{:.1%}", "false-flag share": "{:.1%}"}).hide(axis="index"))
reach = SCAN[(SCAN.sensitivity >= 0.60) & (SCAN["false-flag share"] <= 0.40)]
print(f"Scenarios reaching sensitivity ≥ 60% with false-flag ≤ 40%: {len(reach)} of {len(SCAN)}. "
      "The 70%/25% pair is not jointly reachable here: three binary prior-term signals cannot see the current shock.")
assert (PARAM['beta0']['value'], PARAM['beta']['value']) == (-9.0, 6.0)      # defaults restored

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9)); ax = ax.ravel()
# (a) cumulative attrition by term & tier (from exported tables only)
last_t = TABLES["Enrollments"].groupby("student_id").term.max(); st = TABLES["Students"].set_index("student_id")
for tr_, col in [("high", "#c0392b"), ("typical", "#2471a3")]:
    ids = st.index[st.region_tier == tr_]; wd_t = last_t.loc[ids][TABLES["PersistenceOutcomes"].groupby("student_id").attrition_flag.first().loc[ids]]
    cum = [(wd_t <= t).sum() / len(ids) for t in range(1, T + 1)]
    ax[0].plot(range(1, T + 1), cum, marker="o", color=col, label=tr_)
    ax[0].axhline(P("target_high_2022") if tr_ == "high" else P("target_typical_2022"), color=col, ls=":", lw=1)
ax[0].set(title="(a) Cumulative attrition by term (dotted = calibration target)", xlabel="term", ylabel="cumulative share withdrawn"); ax[0].legend()
# (b) estimators vs truth
e = est.reset_index(drop=True); yy = np.arange(len(e))[::-1]
good = e.estimator.str.contains("Poisson|Mantel")
ax[1].axvline(TRUE_EFFECT, color="k", ls="--", lw=1.2, label="true effect 30%")
ax[1].scatter(e["est. relative reduction"], yy, color=np.where(good, "#1e8449", "#c0392b"), zorder=3)
for i, (lo, hi, g_) in zip(yy, zip(e["CI lo"], e["CI hi"], good)):
    if not np.isnan(lo): ax[1].plot([lo, hi], [i, i], color="#1e8449" if g_ else "#c0392b")
ax[1].set_yticks(yy); ax[1].set_yticklabels(e.estimator, fontsize=7); ax[1].set(title="(b) Estimated relative hazard reduction vs truth", xlabel="relative reduction (1 − RR)"); ax[1].legend()
# (c) share treated by signal count
ax[2].bar(pos.index.astype(str), pos.share_treated, color="#7d3c98")
for i, (s_, n_) in enumerate(zip(pos.share_treated, pos.student_terms)): ax[2].text(i, s_ + .02, f"n={n_:,}", ha="center", fontsize=8)
ax[2].set(title="(c) Assignment is near-deterministic at k=0 and k≥2 (positivity)", xlabel="prior-term signals firing (k)", ylabel="share treated", ylim=(0, 1.12))
# (d) replicate distribution of the MH estimate
ax[3].hist(REP.mh_reduction, bins=10, color="#5d6d7e", edgecolor="white")
ax[3].axvline(TRUE_EFFECT, color="k", ls="--"); ax[3].axvspan(TRUE_EFFECT - window, TRUE_EFFECT + window, color="#1e8449", alpha=.12, label=f"±{window:.1%} window")
ax[3].set(title=f"(d) MH estimate over {len(REP)} independent populations", xlabel="relative reduction"); ax[3].legend()
for a_ in ax: a_.grid(alpha=.25)
fig.suptitle("SYNTHETIC DATA — generator diagnostics (model behaviour, not real students)", fontsize=11, y=1.0)
plt.tight_layout(); fig.savefig(OUT / "diagnostics.png", dpi=130); plt.show()

## 8 · Deliberate edge cases (whitepaper §12)
Both must arise **from the generative mechanism**, not from inserted rows. The code below only *counts* them using the private latent audit; there is no special-case generation path anywhere in §3.

1. **Unflagged high-risk student** — prior-term signals all clean (cell 0 → risk ≈ 0 → no intervention), yet a **shock this term** causes dropout. Consequence of shock ⟂ prior-term observables.
2. **Falsely flagged student** — referred for support on the basis of **one** weak prior-term signal ($k=1$) despite **low underlying vulnerability** ($v<0$, below the cohort mean).

"Non-trivial" is not defined in the whitepaper; the thresholds (≥ 1 % of the relevant event set and ≥ 100 student-terms) are `ASSUMED` (open flag OF-6).

In [ ]:
v_st = L_main["v"][:, None].repeat(T, 1)[m2]; sh_st = L_main["shock"][m2]
kc = ((cell >> 2) & 1) + ((cell >> 1) & 1) + (cell & 1)
e1 = (cell == 0) & ~Tr & sh_st & y
e2 = Tr & (kc == 1) & (v_st < 0)
n_drop, n_treat = int(y.sum()), int(Tr.sum())
share1, share2 = e1.sum() / n_drop, e2.sum() / n_treat
ec = pd.DataFrame([
  {"edge case": "1. Unflagged high-risk (shock-driven dropout, clean record, no support)", "student-terms": int(e1.sum()), "distinct students": int(len(np.unique(sid[e1]))), "share of": "all dropout events", "share": share1},
  {"edge case": "2. Falsely flagged (k=1 weak signal, v<0, supported)",                     "student-terms": int(e2.sum()), "distinct students": int(len(np.unique(sid[e2]))), "share of": "all supported student-terms", "share": share2},
])
display(ec.style.format({"share": "{:.1%}", "student-terms": "{:,}", "distinct students": "{:,}"}).hide(axis="index"))
ok_ec = bool((e1.sum() >= P("edge_min_count")) and (share1 >= P("edge_min_share")) and (e2.sum() >= P("edge_min_count")) and (share2 >= P("edge_min_share")))
SCORE_ALL = pd.concat([SCORE_CORE, pd.DataFrame([{"measure": "Edge-case presence (both cases)", "measured": f"{int(e1.sum()):,} and {int(e2.sum()):,} student-terms",
        "benchmark": f"each ≥ {P('edge_min_count')} and ≥ {P('edge_min_share'):.0%} of its event set", "verdict": "PASS" if ok_ec else "FAIL"}])], ignore_index=True)
print("Reminder: edge case 1 is observable in the export only in part (the shock also shaped this term's own grade/attendance/hold, WP §12).")

In [ ]:
display(Markdown("### Final scorecard — generator, " + f"N={N:,}, seed={CONFIG['MASTER_SEED']}"))
print("⚠", SYNTHETIC_BANNER)
display(SCORE_ALL.style.hide(axis="index").map(lambda v: "color:#1e8449;font-weight:bold" if v == "PASS" else ("color:#b9770e;font-weight:bold" if str(v).startswith(("PASS (","INFO")) else ("color:#c0392b;font-weight:bold" if v == "FAIL" else "")), subset=["verdict"]))
SCORE_ALL.to_csv(OUT / "validation_scorecard.csv", index=False)

## 9 · Customisability demo
`generate_dataset(n_students, seed, ...)` produces a full eight-table dataset for **any** N, reusing the calibrated parameters (no recalibration). Below: three sizes, and a re-run with a different seed to show the output changes but the aggregate behaviour holds (within sampling noise — which is larger at small N).

In [ ]:
def generate_dataset(n_students, seed, delta=None, gamma0=None):
    delta = P("delta") if delta is None else delta; gamma0 = P("gamma0") if gamma0 is None else gamma0
    d = draw_noise(n_students, T, [seed, 0]); L = simulate_latent(d, delta)
    return build_tables(d, L, gamma0)

rows = []
for n_, s_ in [(1_000, 1), (10_000, 2), (100_000, 3), (10_000, 99)]:
    tb, _ = generate_dataset(n_, s_)
    pa = tb["PersistenceOutcomes"].groupby("student_id").attrition_flag.first()
    tr = tb["Students"].set_index("student_id").region_tier
    rows.append({"N students": n_, "seed": s_, **{f"rows: {k}": len(v) for k, v in tb.items() if k in ("Students", "Enrollments", "SupportInterventions")},
                 "attrition high": pa[tr == "high"].mean(), "attrition typical": pa[tr == "typical"].mean(), "pooled": pa.mean()})
print("⚠", SYNTHETIC_BANNER)
display(pd.DataFrame(rows).style.format({"attrition high": "{:.1%}", "attrition typical": "{:.1%}", "pooled": "{:.1%}", "N students": "{:,}"}, thousands=",").hide(axis="index"))

## 10 · Assumptions log, open flags, revision log
### 10.1 Every `ASSUMED` parameter (auto-generated from the registry)

In [ ]:
assumed = param_log[param_log.tag == "ASSUMED"][["parameter", "value", "basis"]].reset_index(drop=True)
print(f"{len(assumed)} ASSUMED parameters — all are candidates for sensitivity analysis (Deliverable 3).")
display(assumed.style.hide(axis="index").set_properties(subset=["basis"], **{"text-align": "left"}))

### 10.2 Open Flags & Revision Log
Statuses: **OPEN** = unresolved, carried forward · **NOTED** = disclosed with a stated consequence · **RESOLVED** = decided here.

| ID | Flag | Status / resolution |
|---|---|---|
| OF-1 | **Realized targeting ≠ Section-8 design targets.** β₀ = −9, β = 6 come from an illustrative submodel and are used *unchanged* (WP Table 4). In the Phase-3 population only ≈ 6 % of student-terms are supported, so realized sensitivity is ≈ 16 % (13–25 % under the alternative definitions tested; hard-capped at ≈ 21 % for a 30 % 'high' prevalence) versus the 70 % target; the false-flag share (≈ 22 %) sits near its 25 % target. The WP's own Table 7 counts imply the same ≈ 5 % supported share. | **RESOLVED — decision: do not re-fit.** The §7.4 scan shows the 70 % / 25 % pair is not jointly reachable: every scenario with sensitivity near 70 % has a false-flag share of ≈ 65 %, because three binary prior-term signals cannot see the current shock. 'Genuinely high current vulnerability' also has no operational definition in the WP, so a re-fit would be ill-posed. Actions: (i) keep (−9, 6) as default — the high-precision regime, and the one every Table 5 result is evidenced on; (ii) treat realized targeting as an *output*, reported in §7.3; (iii) run weaker-targeting / better-overlap settings (e.g. β₀ = −3, β = 2, where all four signal cells contain both arms) as a Deliverable-3 sensitivity analysis. |
| OF-2 | **Held-out tier ratio sits at / just over the band's upper edge.** The single-parameter logit-scale level shift predicts a wider tier gap than the two cohorts showed (WP App. A.3). On this seed the main run lands inside the band (≈ 1.67), but across the 20 replicate populations the ratio averages ≈ 1.69 ± 0.03 and only ≈ 60 % of runs fall inside [1.2, 1.7] — the WP pilot reports ≈ 1.71 ± 0.04 and a FAIL. A single-seed PASS is therefore fragile and flagged as such in the scorecard. | **NOTED.** Not tuned to pass; the band was fixed in the WP before any pilot. The structural cause (one logit-scale shift cannot reproduce how the tier gap moved) is a spec-level limitation. |
| OF-3 | `entry_year` is constant (2018). The model calibrates to one cohort; varying it would be a decoy with no causal role. | **RESOLVED** (ASSUMED). Column kept for schema fidelity. |
| OF-4 | Non-core attribute forms (programme, distance, units, overdue amount, subsidy fields) are not specified by the WP. | **RESOLVED** (all ASSUMED, §2). Programme is causally inert; distance and overdue amount depend weakly on latent vulnerability (permitted by App. B.2 leakage rule). |
| OF-5 | The ±3-pp tolerance is not given a tag in the WP. | **RESOLVED:** tagged LOCKED (design principle, WP §3). |
| OF-6 | "Non-trivial, auditable rate" for the edge cases is undefined in WP Table 5. | **RESOLVED** (ASSUMED): ≥ 1 % of the event set and ≥ 100 student-terms. |
| OF-7 | The 29.4 % national figure is read as a **Class of 2023** value from table layout (WP §10, §14). If that reading is wrong, the held-out design must be redone. | **OPEN** (inherited from the WP). |
| OF-8 | Standardising $v$ within each cohort makes fitted (δ, γ₀) depend slightly on N and seed. `FROZEN_CALIBRATION` is exact only at N = 50,000, seed 193. | **NOTED.** Use `RECALIBRATE=True` for a different N when tier-level fidelity matters. |
| OF-9 | CHED "attrition" includes temporary leave; the model's dropout has no re-entry, so calibrated rates are an upper bound on permanent exit (WP §14). | **NOTED.** |
| OF-10 | Mantel–Haenszel CI uses the Greenland–Robins variance, which treats student-terms as independent; the Poisson estimator uses student-clustered SEs. | **NOTED.** Both are reported; replicate SD sets the point-estimate window. |
| OF-11 | The WP's Table 3 sensitivity/false-flag figures (≈71 % / ≈28 %) apply to an *illustrative* binomial submodel, not to the generated population. | **NOTED** (see OF-1). |
| OF-12 | **IPW results are implementation-dependent.** On identical data, four IPW variants (Hajek/HT × untrimmed/clipped) span roughly −14 % to +74 %. The WP's Table 6 IPW figures (75.6 % untrimmed; −17.8 % trimmed) could not be reproduced exactly because the pilot script was not available; they resemble the HT-clipped and Hajek-clipped variants here. | **NOTED.** Qualitative conclusion (IPW unreliable under near-violated positivity) holds; do not cite a single IPW number. |

**Decision principle used for OF-1/2/7/12:** do not change a calibrated or pre-specified quantity to make a check look better; disclose, label, and move the question to the spec or to sensitivity analysis.

**Revision log** — v1.0.0: first implementation of WP v17 Appendix A; Phase 5b loop re-implemented with CRN and nested bisection; Table 5 checks implemented, including 20-run replication; all analyses run on the exported tables only.

## 11 · What you can do with this dataset (README summary)
The accompanying `README.md` states this in full. In brief, the exported tables support:
1. **Supervised classification** of the per-term `dropout_flag` (note: observables only reveal the latent state noisily).
2. **Discrete-time survival / hazard modelling** on the person-period panel, with right-censoring at term 8.
3. **Causal effect estimation** of the support intervention under confounded assignment — with a known answer (30 % relative hazard reduction) to score against.
4. **Targeting audits** — who is flagged, who is missed (edge case 1), who is flagged unnecessarily (edge case 2).
5. **Positivity and estimator-choice exercises** — why IPW fails here and Mantel–Haenszel / robust-Poisson succeed.

**Do not** report any number from this data as a finding about real Philippine students (WP §13.2).